## Functions and libraries

In [1]:
from fastai.tabular.all import *
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import (
    AdaBoostRegressor, GradientBoostingRegressor,
    BaggingRegressor, RandomForestRegressor
)
from sklearn.tree import DecisionTreeRegressor
from lightgbm import LGBMRegressor
import xgboost as xgb
from catboost import CatBoostRegressor
import re

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

In [2]:
# Función para limpiar nombres de columnas
def clean_column_names(df):
    df = df.copy()
    df.columns = [
        re.sub(r'[^A-Za-z0-9_]+', '_', col)  # deja solo letras, números y "_"
        for col in df.columns
    ]
    return df


In [3]:
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_log_error, mean_squared_error
import numpy as np

# Funciones auxiliares
def mape_percent(y_true, y_pred):
    # MAPE en porcentaje
    return 100 * mean_absolute_percentage_error(y_true, y_pred)

def rmsle(y_true, y_pred):
    # MSLE requiere no-negativos
    y_true_safe = np.maximum(y_true, 0)
    y_pred_safe = np.maximum(y_pred, 0)
    return np.sqrt(mean_squared_log_error(y_true_safe, y_pred_safe))

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Evaluación de modelos
def fit_transform_model(X, y):
    # Split train/test
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Lista de modelos
    models = [
        ("LinearRegression", LinearRegression(n_jobs=-1)),
        ("KNeighborsRegressor", KNeighborsRegressor(n_neighbors=5, n_jobs=-1)),
        ("AdaBoostRegressor", AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42)),
        ("DecisionTreeRegressor", DecisionTreeRegressor(max_depth=10, random_state=42)),
        ("GradientBoostingRegressor", GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)),
        ("BaggingRegressor", BaggingRegressor(n_estimators=50, n_jobs=-1, random_state=42)),
        ("RandomForestRegressor", RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=-1, random_state=42)),
        ("LGBMRegressor", LGBMRegressor(n_estimators=200, learning_rate=0.1, max_depth=-1, n_jobs=-1, random_state=42, verbose=-1)),
        ("XGBRegressor", xgb.XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=6,
                         subsample=0.8, colsample_bytree=0.8, n_jobs=-1,
                         tree_method="hist", random_state=42, verbosity=0)),
        ("CatBoostRegressor", CatBoostRegressor(iterations=200, depth=6, learning_rate=0.1, verbose=False, random_state=42))
    ]
    
    return [(name, model.fit(X_train, y_train).predict(X_test), y_test) 
            for name, model in models]

## Model test with application_train.csv

### Import dataset

In [4]:
train_df = pd.read_parquet("train_2_Prework.parquet", engine='fastparquet')

### Quick prework with fastai

In [5]:
# Cargar el objeto TabularPandas previamente guardado
to = load_pickle('./df_train-tabular-object.pkl')

In [6]:
len(to.train),len(to.valid)

(195580, 48895)

In [7]:
# Una vez hecho el preprocesamiento, se puede ver que los valores del dataframe son todos numéricos.
to.items.head(3)

,start_date,end_date,created_on,l1,l2,l3,l4,rooms,bedrooms,bathrooms,surface_covered,title,description,property_type,covered_ratio,year_created,month_created,age_of_ad,price_m2,distancia_obelisco,indice_urbanidad,rango_distancia,rooms_na,bedrooms_na,bathrooms_na,surface_covered_na,covered_ratio_na
450491,55,449,55,4,26,0,0,-0.019364,-0.136123,3.358834,0.065219,141548,118631,4,-0.019949,-0.908512,0.331889,1.015269,4852.941406,-1.057178,1.316712,>500km,2,2,1,1,1
61573,154,322,154,1,13,175,0,-0.019364,-0.136123,0.409889,-0.012859,101836,162916,1,-0.032294,-0.908512,1.569249,0.070651,452.380951,2.043463,-2.248484,>500km,1,2,1,1,1
656842,186,222,186,1,7,669,0,-0.019364,0.844934,0.409889,-0.002489,28530,110988,1,-0.052916,1.100701,-1.833489,-0.249999,809.248535,-0.280588,0.277704,>500km,2,1,1,1,1


In [8]:
to.items.describe(include='all')  # Para ver estadísticas generales

,start_date,end_date,created_on,l1,l2,l3,l4,rooms,bedrooms,bathrooms,surface_covered,title,description,property_type,covered_ratio,year_created,month_created,age_of_ad,price_m2,distancia_obelisco,indice_urbanidad,rango_distancia,rooms_na,bedrooms_na,bathrooms_na,surface_covered_na,covered_ratio_na
count,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000,2.444750e+05,244475.000000,244475.000000,244475,244475.000000,244475.000000,244475.000000,244475.000000,244475.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,>500km,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,244475,NaN,NaN,NaN,NaN,NaN
mean,159.086365,273.478217,159.086365,1.074482,9.522381,378.976656,128.424338,-0.001193,-0.001313,-0.001357,0.001295,71314.971678,96148.138947,4.230400,-0.000189,0.000468,-0.000325,-0.000439,1.952949e+03,0.000054,-0.000453,NaN,1.279301,1.445870,1.169220,1.142730,1.142730
std,101.183591,144.788460,101.183591,0.462793,10.999082,211.495418,225.998214,1.001551,0.989428,0.996942,1.122574,40904.300614,54196.756571,2.450205,0.962243,1.000047,1.000159,0.999839,4.970811e+03,0.995876,0.991969,NaN,0.448656,0.497062,0.374947,0.349798,0.349798
min,1.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,-1.394259,-4.060351,-0.573093,-0.096427,1.000000,0.000000,1.000000,-0.097083,-0.908512,-1.833489,-1.853250,1.000000e-01,-1.648052,-8.564938,NaN,1.000000,1.000000,1.000000,1.000000,1.000000
25%,68.000000,156.000000,68.000000,1.000000,3.000000,205.000000,0.000000,-0.706811,-0.136123,-0.573093,-0.067758,41988.000000,49532.500000,4.000000,-0.015107,-0.908512,-0.596130,-0.839302,9.714286e+02,-0.298124,0.136727,NaN,1.000000,1.000000,1.000000,1.000000,1.000000
50%,147.000000,287.000000,147.000000,1.000000,7.000000,423.000000,0.000000,-0.019364,-0.136123,-0.573093,-0.052508,63251.000000,100925.000000,4.000000,-0.006484,-0.908512,0.022550,0.148647,1.792453e+03,-0.276262,0.272243,NaN,1.000000,1.000000,1.000000,1.000000,1.000000
75%,240.000000,425.000000,240.000000,1.000000,7.000000,533.000000,194.000000,-0.019364,-0.136123,0.409889,-0.022009,106347.500000,141622.000000,4.000000,0.001510,1.100701,0.950569,0.902608,2.583333e+03,-0.169055,0.299913,NaN,2.000000,2.000000,1.000000,1.000000,1.000000


In [9]:
to.train.xs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 195580 entries, 450491 to 666802
Data columns (total 25 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   start_date          195580 non-null  int16  
 1   end_date            195580 non-null  int16  
 2   created_on          195580 non-null  int16  
 3   l1                  195580 non-null  int8   
 4   l2                  195580 non-null  int8   
 5   l3                  195580 non-null  int16  
 6   l4                  195580 non-null  int16  
 7   title               195580 non-null  int32  
 8   description         195580 non-null  int32  
 9   property_type       195580 non-null  int8   
 10  rooms_na            195580 non-null  int8   
 11  bedrooms_na         195580 non-null  int8   
 12  bathrooms_na        195580 non-null  int8   
 13  surface_covered_na  195580 non-null  int8   
 14  covered_ratio_na    195580 non-null  int8   
 15  rooms               195580 non-nul

### Fit

In [10]:
# Cargar el objeto TabularPandas desde el pickle
to = load_pickle('./df_train-tabular-object.pkl')

# Extraer features y target del conjunto de entrenamiento
X_train = to.train.xs.copy()
y_train = to.train.y.copy()

# Muestreo (opcional, si necesitas trabajar con una muestra más pequeña)
np.random.seed(42)  # Para reproducibilidad
sample_frac = 0.20
sampled_indices = np.random.choice(len(X_train), size=int(len(X_train) * sample_frac), replace=False)
X_train_sample = X_train.iloc[sampled_indices]
y_train_sample = y_train.iloc[sampled_indices]

# Entrenar modelos y obtener predicciones
results = fit_transform_model(X_train_sample, y_train_sample)

# Imprimir resultados una sola vez
print(f"{'Modelo':<25} | {'RMSLE':>10} | {'RMSE':>10}")
print("-" * 50)
for name, y_pred, y_true in results:
    rmsle_val = rmsle(y_true, y_pred)
    rmse_val = root_mean_squared_error(y_true, y_pred)
    print(f"{name:<25} | {rmsle_val:>10.4f} | {rmse_val:>10.4f}")

Modelo                    |      RMSLE |       RMSE
--------------------------------------------------
LinearRegression          |     1.1542 |  2506.7290
KNeighborsRegressor       |     1.0884 |  2687.9761
AdaBoostRegressor         |     1.5074 |  2943.0070
DecisionTreeRegressor     |     0.6157 |  2462.3164
GradientBoostingRegressor |     0.8810 |  2363.1292
BaggingRegressor          |     0.5866 |  2535.4061
RandomForestRegressor     |     0.6436 |  2450.6363
LGBMRegressor             |     1.0993 |  2356.7074
XGBRegressor              |     1.0412 |  2429.9468
CatBoostRegressor         |     0.8987 |  2379.4567
